# SILAS in a notebook

The session is loaded when the notebook's kernel starts (kernel **SILAS**): every command of the
terminal session (`bt`) is there, e.g. `show_scans()`, `analyse_scan(...)`, `commands()`.
Plots appear below the cells; the viewers (`start_mpccd_viewer`, `start_tt_viewer`,
`view_full_frame`, ...) appear as interactive figures: click on a figure, then use its keys
(`h` for help in the MPCCD and time-tool viewers, `q` stops a viewer).

This file is part of the code (`~/SILAS/notebooks/`) and is replaced when the code is synced:
**save your own notebooks somewhere else** (e.g. `/UserData/<user>/notebooks/`).

In [ ]:
# only needed for another experiment config (or in a kernel without the session):
# from silas.cli.notebook import start
# start(config="config/other.json")

## What is there

In [ ]:
commands()          # every command with its first line of help; help(name) for the full text

In [ ]:
show_scans()        # the scans of the run log

## Examples

Remove the `#` of the line you want to run.

In [ ]:
# scan_info(-1)                                    # the newest scan
# analyse_scan(-1, max_tags=100)                   # quick look at the ROI analysis: the plot appears below
# d = load_scan(-1)                                # its results as pandas tables

In [ ]:
# start_mpccd_viewer(run=1234501, max_tags=300)    # offline MPCCD view of a stored run (interactive figure)
# start_tt_viewer(run=1234501)                     # the time-tool camera, shot by shot

In [ ]:
# ff = load_full_frame(-1)                          # a full-frame reduction (full_frame_job makes it)
# view_full_frame(-1)                               # pump on, off and on / off of a step (keys: arrows, r, m, d, b)

### Scan ROI analysis

In [ ]:
# analyse_scan(-1)                                  # analyse the newest scan
# analyse_scan(1234560, tt=True, bin_ps=0.02)       # analyse a specific scan, with time-tool binning

In [ ]:
# scan_results(-1)                      # the results of the newest scan, as pandas tables
# scan_results(1234560, show=True)      # specific scan, show the plots as well

In [ ]:
## Loading using silas
import matplotlib.pyplot as plt

from silas.offline.scan_analysis import load_h5

# Specific h5 file
data = load_h5("scan_123456.h5")
# or
# data = load_scan(123456)

res = data["results"]

roi = res["rois"][0]      # first ROI

x = res["raw"]["x"]
med = res["raw"]["med"][0, :, 0]
lo = res["raw"]["lo"][0, :, 0]
hi = res["raw"]["hi"][0, :, 0]

plt.errorbar(
    x,
    med,
    yerr=[lo, hi],
    marker="o",
    capsize=2,
)

plt.xlabel("Scan position")
plt.ylabel("ROI signal")
plt.title(roi)
plt.grid(True)
plt.show()

In [ ]:
## loading with silas and time tool binning
import matplotlib.pyplot as plt

from silas.offline.scan_analysis import load_h5

# Specific h5 file
data = load_h5("scan_123456.h5")
# or
# data = load_scan(123456)

tt = data["timetool"]

roi = tt["panels"][0][1][0]     # first ROI column

bins = tt["bins"]

x = bins["t_ps"]

med = bins[roi + "_median"]
lo = med - bins[roi + "_q16"]
hi = bins[roi + "_q84"] - med

plt.errorbar(
    x,
    med,
    yerr=[lo, hi],
    marker="o",
    capsize=2,
)

plt.xlabel("Time (ps)")
plt.ylabel("ROI signal")
plt.title(roi)
plt.grid(True)
plt.show()

In [ ]:
## No dependecies on silas, just h5py and pandas
import h5py
import matplotlib.pyplot as plt

with h5py.File("scan_123456.h5") as f:

    raw = f["results"]["raw"]

    x = raw["x"][:]
    med = raw["med"][0, :, 0]
    lo = raw["lo"][0, :, 0]
    hi = raw["hi"][0, :, 0]

plt.errorbar(
    x,
    med,
    yerr=[lo, hi],
    marker="o",
    capsize=2,
)

plt.xlabel("Scan position")
plt.ylabel("ROI signal")
plt.grid(True)
plt.show()

In [ ]:
## No dependecies on silas, just h5py and pandas
## and time tool binning
import h5py
import matplotlib.pyplot as plt

with h5py.File("scan_123456.h5") as f:

    bins = f["timetool"]["bins"]

    # choose any ROI column
    roi = "roi_my_roi"

    x = bins["t_ps"][:]

    med = bins[roi + "_median"][:]
    lo = med - bins[roi + "_q16"][:]
    hi = bins[roi + "_q84"][:] - med

plt.errorbar(
    x,
    med,
    yerr=[lo, hi],
    marker="o",
    capsize=2,
)

plt.xlabel("Time (ps)")
plt.ylabel("ROI signal")
plt.grid(True)
plt.show()

### Full-frame analysis

Launch the full-frame reduction jobs in the que on the cluster. In the notebook, you can analyse the results.

In [ ]:
full_frame_job(1234501)                             # submits: 16 cores, memory estimated
full_frame_job(1234501, panels=[7, 8], tt=True)     # options as reduce_full_frame
full_frame_job(1234501, cores=32, mem_gb=96)        # more of the node
full_frame_job(1234501, submit=False)               # only print the qsub command

In [ ]:
## Viewing the full-frame reduction results
view_full_frame(1234501)                           # the reduction of the chosen panels (set_ff_panels)
view_full_frame(1234501, step=10, ref="i0_1")      # where it starts (default: the first reference)
view_full_frame(1234501, ref="none")               # start without a reference
view_full_frame("~/data/fullframe_1234501.h5")     # any file, e.g. a copy at home

In [ ]:
## Opening the full-frame reduction results in code using silas
from silas.offline.fullframe import FullFrame
ff = FullFrame("fullframe_1234501_p78.h5")         # in the session: load_full_frame(1234501)
ff.steps                                           # the step table (pandas)
ff.image("raw", step=0, state="off", stat="mean")  # (panels, rows, cols)
ff.image("i0_1", step=10, state="on")              # pixel / i0_1 per shot, median
ff.onoff("raw", step=10)                           # (ratio, lower error, upper error)
ff.difference("raw", step=10)                      # (on - off, lower error, upper error)
ff.step_referenced("i0_1", step=10)                # median pixel / median i0_1 (step_reference)
ff.image("pairs_i0_1", step=10)                    # nearest-off pairs (pairs); "pairs_diff_i0_1": on - off
ff.bins                                            # the bin table; ff.image("onoff_i0_1", bin=3), ff.image("diff", bin=3)
ff.stack("raw", state="on", stat="mean")           # all steps: (steps, panels, rows, cols)
ff.assemble(ff.image("raw", step=10))              # 2-D image in detector geometry (gaps NaN)
ff.close()

In [ ]:
## Opening the full-frame reduction results in code without silas